In [ ]:
# notebook to find multiTF enhancer instances where gtex eQTLs overlap multiTF enhancers

In [1]:
# import packages
import pandas as pd
import os
from tqdm import tqdm
import numpy as np
import gc
import psutil
from collections import Counter
import pickle

In [2]:
# open summary multiTF data
all_dELS_emVars = pd.read_csv('results_final/multiTF_variants.tsv', sep = '\t')
# filter for only those in multiTF enhancers
multiTF_emVars = all_dELS_emVars[all_dELS_emVars['is_multiTF'] == True].copy()
# add gtex id
multiTF_emVars.loc[:,'gtex_id'] = [('_').join([chrom, str(pos), ref, alt]) + '_b38' for chrom, pos, ref, alt in zip(multiTF_emVars['chrom'],
                                                                                                                    multiTF_emVars['pos'],
                                                                                                                    multiTF_emVars['ref'],
                                                                                                                    multiTF_emVars['alt'])]

In [3]:
# load padding-zone emVars (4bp flanking filtered seqlets, not captured by exact BED intersection)
padding_emVars_path = 'results_final/padding_emVars_gnomAD_v4.tsv'
if os.path.exists(padding_emVars_path):
    padding_emVars = pd.read_csv(padding_emVars_path, sep='\t')
    padding_cols = ['variant_id', 'chrom', 'pos', 'ref', 'alt', 'cell_type', 'skew_pred', 'enhancer_ids', 'is_multiTF']
    multiTF_emVars = pd.concat(
        [multiTF_emVars, padding_emVars[padding_cols]], ignore_index=True
    ).drop_duplicates(subset=['variant_id', 'cell_type'], keep='first')
    # recompute gtex_id for new variants
    multiTF_emVars.loc[:,'gtex_id'] = [
        ('_').join([chrom, str(pos), ref, alt]) + '_b38'
        for chrom, pos, ref, alt in zip(
            multiTF_emVars['chrom'], multiTF_emVars['pos'],
            multiTF_emVars['ref'], multiTF_emVars['alt']
        )
    ]
    print(f'Added {len(padding_emVars)} padding emVars → {len(multiTF_emVars)} total multiTF emVars')
else:
    print(f'Padding emVars file not found: {padding_emVars_path} — skipping')

Padding emVars file not found: results_final/padding_emVars_gnomAD_v4.tsv — skipping


In [4]:
# open all gtex data
# define the path to the fine mapped variants as a variable
path2fineMapped = '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/promoter_sat_mut_comp/raw_data/gtex_v10_fineMapping/SuSiE_fineMapped'
# define the path to the gtex info as a variable
path2gtex_v10 = '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/promoter_sat_mut_comp/raw_data/gtex_v10_fineMapping/GTEx_Analysis_v10_eQTL_updated'

In [5]:
# iterate through all fine mapped variants and collapse into a single DF
fineMapped2cat = []
fineMappedDict = {}
for i in tqdm([i for i in os.listdir(path2fineMapped) if i.endswith('.parquet')]):
    parqLife = pd.read_parquet(f'{path2fineMapped}/{i}')
    # add tissue column
    parqLife.loc[:,'tissue'] = [i.split('.')[0] for j in range(len(parqLife))]                                                                                                                                                                 
    fineMapped2cat.append(parqLife)
    fineMappedDict.update({
        i.split('.')[0] : parqLife
    })
# make df
fineMapped_all = pd.concat(fineMapped2cat)

  2%|▏         | 1/50 [00:00<00:42,  1.16it/s]

100%|██████████| 50/50 [00:17<00:00,  2.86it/s]


In [6]:
len(fineMapped_all['variant_id'].unique())

1934263

In [7]:
multiTF_emVars.head()

,variant_id,chrom,pos,ref,alt,cell_type,skew_pred,enhancer_ids,is_multiTF,gtex_id
375,chr10:100044642:C:A,chr10,100044642,C,A,k562,0.550848,EH38E2919982,True,chr10_100044642_C_A_b38
376,chr10:100044642:C:G,chr10,100044642,C,G,k562,0.638601,EH38E2919982,True,chr10_100044642_C_G_b38
377,chr10:100044644:G:A,chr10,100044644,G,A,k562,0.574964,EH38E2919982,True,chr10_100044644_G_A_b38
378,chr10:100044644:G:T,chr10,100044644,G,T,k562,0.511102,EH38E2919982,True,chr10_100044644_G_T_b38
379,chr10:100044646:G:A,chr10,100044646,G,A,k562,0.561417,EH38E2919982,True,chr10_100044646_G_A_b38


In [8]:
multiTF_GTEx = {}
# iterate through finemapping data and annotate predictions/multiTF variants
for tissue in fineMappedDict.keys():
    # get the GTEx data for that tissue
    tissueData = fineMappedDict.get(tissue)
    # filter all emVars for those in tissue Data
    tissueMultiTFVars = multiTF_emVars[multiTF_emVars['gtex_id'].isin(tissueData['variant_id'].tolist())]
    # merge data
    mergedVars = tissueData.merge(tissueMultiTFVars, left_on='variant_id', right_on='gtex_id', how='inner')
    multiTF_GTEx.update({
        tissue : mergedVars
    })

In [9]:
# iterate through each tissue and filter for the high PIP variants for each tissue as our high priority checks
highPIP_multiTF_GTEx2cat = []
for tissue in multiTF_GTEx.keys():
    # get the tissue level data
    tissueOverlap = multiTF_GTEx.get(tissue)
    # add tissue to df for concatentating
    tissueOverlap.loc[:, 'tissue'] = [tissue for i in range(len(tissueOverlap))]
    # filter for high PIP (> 0.9)
    highPipOverlap = tissueOverlap[tissueOverlap['pip'] > 0.9].copy()
    # append to list for concatenating
    highPIP_multiTF_GTEx2cat.append(highPipOverlap)
# concatenate all high PIP variants
highPIP_multiTF_GTEx = pd.concat(highPIP_multiTF_GTEx2cat)

In [10]:
len(highPIP_multiTF_GTEx['variant_id_y'].unique())

90

In [11]:
# filter DF to avoid redundancy
highPip2save = highPIP_multiTF_GTEx.filter([
    'phenotype_id', 'gene_name', 'biotype', 'pip', 'af',
    'cs_id', 'cs_size', 'afc', 'afc_se', 'tissue',
    'chrom', 'pos', 'ref', 'alt', 'cell_type', 'skew_pred',
    'enhancer_ids', 'is_multiTF', 'gtex_id'
])
highPip2save.loc[:, 'variant_id'] = [(':').join(i.split('_')[0:-1]) for i in highPip2save['gtex_id']]

In [12]:
highPip2save.head()

,phenotype_id,gene_name,biotype,pip,af,cs_id,cs_size,afc,afc_se,tissue,chrom,pos,ref,alt,cell_type,skew_pred,enhancer_ids,is_multiTF,gtex_id,variant_id
0,ENSG00000215915.10,ATAD3C,protein_coding,0.994196,0.254571,1,1,-1.196924,0.099390,Adipose_Subcutaneous,chr1,1440430,C,G,sknsh,-1.769541,EH38E2777558,True,chr1_1440430_C_G_b38,chr1:1440430:C:G
4,ENSG00000116688.18,MFN2,protein_coding,0.999149,0.454290,1,1,-0.086193,0.023958,Adipose_Subcutaneous,chr1,11986032,G,A,hepg2,-2.147629,EH38E1319284,True,chr1_11986032_G_A_b38,chr1:11986032:G:A
38,ENSG00000180767.11,CHST13,protein_coding,1.000000,0.296765,1,1,-1.989094,0.126096,Adipose_Subcutaneous,chr3,126530422,G,A,k562,-1.450078,EH38E3536992,True,chr3_126530422_G_A_b38,chr3:126530422:G:A
45,ENSG00000109689.19,STIM2,protein_coding,0.959521,0.254571,2,1,-0.091416,0.033677,Adipose_Subcutaneous,chr4,26826816,C,T,k562,-1.479022,EH38E3578473,True,chr4_26826816_C_T_b38,chr4:26826816:C:T
46,ENSG00000109689.19,STIM2,protein_coding,0.959521,0.254571,2,1,-0.091416,0.033677,Adipose_Subcutaneous,chr4,26826816,C,T,hepg2,-1.278330,EH38E3578473,True,chr4_26826816_C_T_b38,chr4:26826816:C:T


In [13]:
### phase 2 ###
# get all other emVars in the enhancers in the other TF/in the same TF to find instances where allele frequencies differ.
# for now let's annotate with gnomAD v3 (from Stephen's analysis) - check into v4 shortly
# open the pickle
with open('results_final/multiTF_analysis.pkl', 'rb') as f:
    multiTF_pickle = pickle.load(f)
# open the multiTF summary file
multiTF_summary = pd.read_csv('results_final/multiTF_summary.tsv', sep = '\t')
# make a dictionary of enhancer ID : lead variant pairs
enhID_leadVarDict = dict(zip(highPip2save['enhancer_ids'], highPip2save['gtex_id']))
# go back the other way - leadVariant to enhancer
leadVar_enhID = dict(zip(highPip2save['gtex_id'], highPip2save['enhancer_ids']))

In [14]:
  # Pre-group for O(1) lookups instead of filtering the full df each time
  # let's add a 4 bp pad to seqlets to see if we can capture some additional variants/TF relevant variation                                                                                                                                                                                                                                            
  emvar_groups = multiTF_emVars.groupby(['cell_type', 'enhancer_ids'])                                                                                                                                                                                                                                                
                                                                                                                                                                                                                                                                                                                      
  allSeqletEmvars2cat = []                                                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                                                      
  for leadVar in tqdm(highPip2save['gtex_id'].unique()):                                                                                                                                                                                                                                            
      enhancer = leadVar_enhID.get(leadVar)                                                                                                                                                                                                                                                                           
      enhancerCells = list(multiTF_summary[multiTF_summary['enhancer_id'] == enhancer]['cell_type'].unique())                                                                                                                                                                                                         
                                                                                                                                                                                                                                                                                                                      
      for cell in enhancerCells:                                                                                                                                                                                                                                                                                      
          filteredSeqlets = multiTF_pickle['multiTF_enhancers'][cell][enhancer]['filtered_seqlets']                                                                                                                                                                                                                   
                                                                                                                                                                                                                                                                                                                      
          # Get all variants for this cell/enhancer combo once                                                                                                                                                                                                                                                        
          try:                                                                                                                                                                                                                                                                                                        
              cell_enh_vars = emvar_groups.get_group((cell, enhancer))                                                                                                                                                                                                                                                
          except KeyError:                                                                                                                                                                                                                                                                                            
              continue                                                                                                                                                                                                                                                                                                
                                                                                                                                                                                                                                                                                                                      
          cell_seqlet_vars = []                                                                                                                                                                                                                                                                                       
          tfCount = 1                                                                                                                                                                                                                                                                                                 
          lead_found = False                                                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                                      
          for start, end, repTF, contrib in zip(filteredSeqlets['start'], filteredSeqlets['end'],                                                                                                                                                                                                                     
                                                 filteredSeqlets['vierstra_cluster'], filteredSeqlets['rep_tf_contrib']):                                                                                                                                                                                            
              # Now just filter by position (already filtered by cell/enhancer)                                                                                                                                                                                                                                       
              seqletVars = cell_enh_vars[(cell_enh_vars['pos'] >= start - 4) & (cell_enh_vars['pos'] <= end + 4)].copy() # 4bp pad added 2/6/26, for initial no pad analysis just drop '+/- 4'                                                                                                                                                                                                
                                                                                                                                                                                                                                                                                                                      
              if len(seqletVars) == 0:                                                                                                                                                                                                                                                                                
                  tfCount += 1                                                                                                                                                                                                                                                                                        
                  continue                                                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                                                      
              # Scalar assignment is faster than list comprehension                                                                                                                                                                                                                                                   
              seqletVars['tf_family'] = repTF                                                                                                                                                                                                                                                                         
              seqletVars['tf_contrib'] = contrib                                                                                                                                                                                                                                                                      
              seqletVars['tf_instance'] = tfCount                                                                                                                                                                                                                                                                     
              seqletVars['leadVar'] = leadVar                                                                                                                                                                                                                                                                     
              seqletVars['is_leadVariant'] = (seqletVars['gtex_id'] == leadVar).astype(int)                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                                                                                      
              if seqletVars['is_leadVariant'].sum() > 0:                                                                                                                                                                                                                                                              
                  lead_found = True                                                                                                                                                                                                                                                                                   
                                                                                                                                                                                                                                                                                                                      
              cell_seqlet_vars.append(seqletVars)                                                                                                                                                                                                                                                                     
              tfCount += 1                                                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                                                      
          # Only concat and append if lead variant was found in this cell's seqlets                                                                                                                                                                                                                                   
          if lead_found and cell_seqlet_vars:                                                                                                                                                                                                                                                                         
              allSeqletEmvars2cat.append(pd.concat(cell_seqlet_vars).sort_values(by='pos'))                                                                                                                                                                                                                           
                                                                                                                                                                                                                                                                                                                      
  allSeqletEmVars = pd.concat(allSeqletEmvars2cat, ignore_index=True)

  0%|          | 0/90 [00:00<?, ?it/s]

100%|██████████| 90/90 [00:08<00:00, 11.25it/s]


In [15]:
len(allSeqletEmVars)

2046

In [16]:
allSeqletEmVars.head()

,variant_id,chrom,pos,ref,alt,cell_type,skew_pred,enhancer_ids,is_multiTF,gtex_id,tf_family,tf_contrib,tf_instance,leadVar,is_leadVariant
0,chr1:11986031:C:A,chr1,11986031,C,A,hepg2,-0.811828,EH38E1319284,True,chr1_11986031_C_A_b38,ETS/1,11.187316,1,chr1_11986032_G_A_b38,0
1,chr1:11986031:C:G,chr1,11986031,C,G,hepg2,-1.366258,EH38E1319284,True,chr1_11986031_C_G_b38,ETS/1,11.187316,1,chr1_11986032_G_A_b38,0
2,chr1:11986031:C:T,chr1,11986031,C,T,hepg2,-2.161320,EH38E1319284,True,chr1_11986031_C_T_b38,ETS/1,11.187316,1,chr1_11986032_G_A_b38,0
3,chr1:11986032:G:A,chr1,11986032,G,A,hepg2,-2.147629,EH38E1319284,True,chr1_11986032_G_A_b38,ETS/1,11.187316,1,chr1_11986032_G_A_b38,1
4,chr1:11986032:G:C,chr1,11986032,G,C,hepg2,-2.189756,EH38E1319284,True,chr1_11986032_G_C_b38,ETS/1,11.187316,1,chr1_11986032_G_A_b38,0


In [17]:
  import subprocess
                                                                                                                                                                             
  af_pop_cols = ['AF', 'AF_afr', 'AF_ami', 'AF_amr', 'AF_asj', 'AF_eas',
                 'AF_fin', 'AF_mid', 'AF_nfe', 'AF_remaining', 'AF_sas']

  af_matches = []

  for chrom in tqdm(allSeqletEmVars['chrom'].unique()):
      # Get positions for this chromosome
      chrom_vars = allSeqletEmVars[allSeqletEmVars['chrom'] == chrom]
      positions = chrom_vars['pos'].unique()

      # Build grep pattern for positions (match at start of line after chrom)
      pattern = '|'.join([f'^{chrom}\t{pos}\t' for pos in positions])

      filepath = f'/projects/tewhey-lab/buttsj/Variant_Effects/gnomad/gnomad_v4/filtered_data/popLevel/{chrom}_gnomAD_v4_pass_popLevel_af.tsv.gz'

      # Get header first
      header_cmd = f"zcat {filepath} | head -1"
      header = subprocess.run(header_cmd, shell=True, capture_output=True, text=True).stdout.strip().split('\t')

      # Use zgrep to extract matching lines
      grep_cmd = f"zcat {filepath} | grep -E '{pattern}'"
      result = subprocess.run(grep_cmd, shell=True, capture_output=True, text=True)

      if result.stdout:
          from io import StringIO
          chunk_df = pd.read_csv(StringIO(result.stdout), sep='\t', names=header, na_values='.')
          chunk_df['id'] = (chunk_df['CHROM'] + '_' + chunk_df['POS'].astype(str) + '_' +
                           chunk_df['REF'] + '_' + chunk_df['ALT'] + '_b38')
          af_matches.append(chunk_df[['id'] + af_pop_cols])

  all_af_data = pd.concat(af_matches, ignore_index=True)

  GTEx_alleleSeqletEmVars_AF = allSeqletEmVars.merge(
      all_af_data, left_on='gtex_id', right_on='id', how='left'
  ).drop(columns=['id']).rename(columns={'AF': 'af'})

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [16:26<00:00, 49.33s/it]


In [18]:
GTEx_alleleSeqletEmVars_AF.head()

,variant_id,chrom,pos,ref,alt,cell_type,skew_pred,enhancer_ids,is_multiTF,gtex_id,...,AF_afr,AF_ami,AF_amr,AF_asj,AF_eas,AF_fin,AF_mid,AF_nfe,AF_remaining,AF_sas
0,chr1:11986031:C:A,chr1,11986031,C,A,hepg2,-0.811828,EH38E1319284,True,chr1_11986031_C_A_b38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chr1:11986031:C:G,chr1,11986031,C,G,hepg2,-1.366258,EH38E1319284,True,chr1_11986031_C_G_b38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chr1:11986031:C:T,chr1,11986031,C,T,hepg2,-2.161320,EH38E1319284,True,chr1_11986031_C_T_b38,...,0.000000,0.000000,0.000065,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
3,chr1:11986032:G:A,chr1,11986032,G,A,hepg2,-2.147629,EH38E1319284,True,chr1_11986032_G_A_b38,...,0.111681,0.325658,0.417158,0.363033,0.411662,0.649423,0.47619,0.524231,0.418406,0.474917
4,chr1:11986032:G:C,chr1,11986032,G,C,hepg2,-2.189756,EH38E1319284,True,chr1_11986032_G_C_b38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
GTEx_alleleSeqletEmVars_AF['af'].isna().value_counts()

af
True     1854
False     192
Name: count, dtype: int64

In [20]:
  gtex_phenotype_info = highPip2save[[                                                                                                                                                                                                                                                                       
      'gtex_id',                                                                                                                                                                                                                                                                                             
      'phenotype_id',                                                                                                                                                                                                                                                                                        
      'gene_name',                                                                                                                                                                                                                                                                                           
      'biotype',                                                                                                                                                                                                                                                                                             
      'tissue',                                                                                                                                                                                                                                                                                              
      'pip',
      'af',                                                                                                                                                                                                                                                                                                 
      'afc',           # allelic fold change (effect size)                                                                                                                                                                                                                                                   
      'afc_se',                                                                                                                                                                                                                                                                                              
      'cs_id',         # credible set ID                                                                                                                                                                                                                                                                     
      'cs_size'                                                                                                                                                                                                                                                                                              
  ]].drop_duplicates()                                                                                                                                                                                                                                                                                       
                                                                                                                                                                                                                                                                                                             
  # Check cardinality                                                                                                                                                                                                                                                                                        
  print(f"Unique lead variants (gtex_id): {gtex_phenotype_info['gtex_id'].nunique()}")                                                                                                                                                                                                                       
  print(f"Total variant-gene-tissue combinations: {len(gtex_phenotype_info)}")                                                                                                                                                                                                                               
                                                                                                                                                                                                                                                                                                             
  # Merge onto allele frequency data                                                                                                                                                                                                                                                                         
  GTEx_alleleSeqletEmVars_AF_pheno = GTEx_alleleSeqletEmVars_AF.merge(                                                                                                                                                                                                                                       
      gtex_phenotype_info,                                                                                                                                                                                                                                                                                   
      left_on='leadVar',                                                                                                                                                                                                                                                                                     
      right_on='gtex_id',                                                                                                                                                                                                                                                                                    
      how='left',                                                                                                                                                                                                                                                                                            
      suffixes=('', '_eqtl')                                                                                                                                                                                                                                                                                 
  )                                                                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                             
  print(f"Rows before merge: {len(GTEx_alleleSeqletEmVars_AF)}")                                                                                                                                                                                                                                             
  print(f"Rows after merge: {len(GTEx_alleleSeqletEmVars_AF_pheno)}")      

Unique lead variants (gtex_id): 90
Total variant-gene-tissue combinations: 211
Rows before merge: 2046
Rows after merge: 6229


In [21]:
GTEx_alleleSeqletEmVars_AF_pheno.keys()

Index(['variant_id', 'chrom', 'pos', 'ref', 'alt', 'cell_type', 'skew_pred',
       'enhancer_ids', 'is_multiTF', 'gtex_id', 'tf_family', 'tf_contrib',
       'tf_instance', 'leadVar', 'is_leadVariant', 'af', 'AF_afr', 'AF_ami',
       'AF_amr', 'AF_asj', 'AF_eas', 'AF_fin', 'AF_mid', 'AF_nfe',
       'AF_remaining', 'AF_sas', 'gtex_id_eqtl', 'phenotype_id', 'gene_name',
       'biotype', 'tissue', 'pip', 'af_eqtl', 'afc', 'afc_se', 'cs_id',
       'cs_size'],
      dtype='object')

In [22]:
# save high pip variants to disk for downstream analysis
# GTEx_alleleSeqletEmVars_AF_pheno.to_csv(
#     'GTEx_MultiTF_emVars_HighPIP_gnomAD_v4_annotated.tsv',
#     sep = '\t',
#     index = False
# )

In [23]:
# annotate all emVars with lead variant skew comparison (no phenocopy filter - keep all rows)
# Add absolute effect                                                                                                                                                                                                                                               
GTEx_alleleSeqletEmVars_AF_pheno['abs_skew'] = GTEx_alleleSeqletEmVars_AF_pheno['skew_pred'].abs()                                                                                                                                                                  
                                                                                                                                                                                                                                                                      
group_cols = ['enhancer_ids', 'cell_type', 'phenotype_id', 'tissue']                                                                                                                                                                                                
                                                                                                                                                                                                                                                                      
# Get lead variant effect per group                                                                                                                                                                                                                                 
lead_effects = (GTEx_alleleSeqletEmVars_AF_pheno[GTEx_alleleSeqletEmVars_AF_pheno['is_leadVariant'] == 1]                                                                                                                                                           
                  .groupby(group_cols)['abs_skew']                                                                                                                                                                                                                    
                  .first()                                                                                                                                                                                                                                            
                  .rename('lead_abs_skew'))                                                                                                                                                                                                                           
                                                                                                                                                                                                                                                                      
# Get max effect per group                                                                                                                                                                                                                                          
max_effects = (GTEx_alleleSeqletEmVars_AF_pheno                                                                                                                                                                                                                     
                 .groupby(group_cols)['abs_skew']                                                                                                                                                                                                                     
                 .max()                                                                                                                                                                                                                                               
                 .rename('max_abs_skew'))                                                                                                                                                                                                                             
                                                                                                                                                                                                                                                                      
# Combine lead and max effects
effect_comparison = pd.concat([lead_effects, max_effects], axis=1).dropna()                                                                                                                                                                                         
effect_comparison['skew_ratio'] = effect_comparison['max_abs_skew'] / effect_comparison['lead_abs_skew']                                                                                                                                                            
                                                                                                                                                                                                                                                                      
# Merge annotations onto ALL variants (no filtering by max > lead)
all_emVars_annotated_gtex = GTEx_alleleSeqletEmVars_AF_pheno.merge(                                                                                                                                                                                                        
      effect_comparison.reset_index(),                                                                                                                                                                                                                                
      on=group_cols,                                                                                                                                                                                                                                                  
      how='inner'                                                                                                                                                                                                                                                     
)
# add column flagging if this variant's abs_skew exceeds the lead variant's abs_skew
all_emVars_annotated_gtex['exceeds_lead_skew'] = all_emVars_annotated_gtex['abs_skew'] > all_emVars_annotated_gtex['lead_abs_skew']

print(f'Total annotated rows: {len(all_emVars_annotated_gtex)}')
print(f'Contexts where lead is not strongest: {effect_comparison[effect_comparison["max_abs_skew"] > effect_comparison["lead_abs_skew"]].shape[0]}')
print(f'Variants exceeding lead skew: {all_emVars_annotated_gtex["exceeds_lead_skew"].sum()}')

Total annotated rows: 6229
Contexts where lead is not strongest: 146
Variants exceeding lead skew: 4266


In [24]:
all_emVars_annotated_gtex.to_csv('GTEx_MultiTF_allEmVars_HighPIP_gnomAD_v4_annotated_4bp_pad.tsv', sep = '\t', index = False)

In [ ]:
### SCRATCH BELOW ###

In [70]:
# open the multiTF summary file
multiTF_summary = pd.read_csv('results_final/multiTF_summary.tsv', sep = '\t')
# filter for GATA and K562
gata_k562_multiTFs = multiTF_summary[(multiTF_summary['cell_type'] == 'k562') & (multiTF_summary['multiTF_clusters'] == 'GATA')]

In [71]:
multiTF_summary[multiTF_summary['enhancer_id'] == 'EH38E1319284']

,enhancer_id,cell_type,multiTF_clusters,n_multiTF_clusters,n_filtered_seqlets,n_all_seqlets,n_emvars,chrom,start,end
25748,EH38E1319284,hepg2,ETS/1,1,2,4,60,chr1,11986028,11986216


In [72]:
gata_k562_multiTFs

,enhancer_id,cell_type,multiTF_clusters,n_multiTF_clusters,n_filtered_seqlets,n_all_seqlets,n_emvars,chrom,start,end
109,EH38E3504978,k562,GATA,1,2,11,215,chr3,33936383,33936600
149,EH38E2559337,k562,GATA,1,2,12,201,chr7,64614909,64615055
169,EH38E1971301,k562,GATA,1,2,13,197,chr2,8927591,8927836
418,EH38E2066366,k562,GATA,1,2,9,170,chr2,201901905,201902046
494,EH38E1563752,k562,GATA,1,3,12,166,chr11,97963707,97963999
...,...,...,...,...,...,...,...,...,...,...
126465,EH38E1750531,k562,GATA,1,2,2,14,chr15,30962182,30962200
126770,EH38E3557221,k562,GATA,1,2,4,14,chr3,183184450,183184582
132232,EH38E2899262,k562,GATA,1,2,2,11,chr10,48285121,48285139
153847,EH38E2038477,k562,GATA,1,2,5,6,chr2,144700033,144700211


In [73]:
# let's take a first pass at whole blood eQTLS in K562 in GATA (easiest example i suppose)
# get the whole blood multiTF vars
wholeBlood_multiTF = multiTF_GTEx['Whole_Blood'].sort_values(by='cs_id', ascending=True)
wholeBlood_multiTF_K562 = wholeBlood_multiTF[wholeBlood_multiTF['cell_type'] == 'k562'].sort_values(by='pip', ascending=False)
# filter those for the multiTF enhancers
wholeBlood_multiTF_K562_GATAs = wholeBlood_multiTF_K562[wholeBlood_multiTF_K562['enhancer_ids'].isin(gata_k562_multiTFs['enhancer_id'].tolist())]

In [75]:
wholeBlood_multiTF_K562_GATAs.keys()

Index(['phenotype_id', 'gene_name', 'biotype', 'variant_id_x', 'pip', 'af',
       'cs_id', 'cs_size', 'afc', 'afc_se', 'tissue', 'variant_id_y', 'chrom',
       'pos', 'ref', 'alt', 'cell_type', 'skew_pred', 'enhancer_ids',
       'is_multiTF', 'gtex_id'],
      dtype='object')

In [ ]:
fineMappedDict['Whole_Blood'][fineMappedDict['Whole_Blood']['phenotype_id'] == 'ENSG00000187699.10']

In [ ]:
multiTF_GTEx['Whole_Blood'][(multiTF_GTEx['Whole_Blood']['pip'] > 0.9) & (multiTF_GTEx['Whole_Blood']['cell_type'] == 'k562')]

In [ ]:
len(multiTF_GTEx['Whole_Blood'][(multiTF_GTEx['Whole_Blood']['pip'] > 0.9) & (multiTF_GTEx['Whole_Blood']['cell_type'] == 'k562')]['gtex_id'].unique())

In [ ]:
multiTF_GTEx['Whole_Blood'][(multiTF_GTEx['Whole_Blood']['pip'] > 0.9) & (multiTF_GTEx['Whole_Blood']['cell_type'] == 'k562')].keys()

In [ ]:
multiTF_summary[multiTF_summary['enhancer_id'] =='EH38E3536992']

In [ ]:
multiTF_summary[multiTF_summary['enhancer_id'] == 'EH38E3480167']